# 🔍 Search in PDF, Semantically, with Proof

**Citation-Aware RAG Pipeline** — Not "chat with PDF", but **search with proof**.

This notebook handles PDFs with diverse content: text, tables, images, diagrams, multi-column layouts, LaTeX, and scanned documents.

**Key Features:**
- 📄 **Docling** for intelligent PDF extraction (layout, OCR, tables, figures)
- ✂️ **HybridChunker** with full citation provenance (page, bbox, headings)
- 🖼️ **Multimodal VLM** for image-to-text summaries
- 🔀 **Hybrid Search** — BM25 + Vector with Reciprocal Rank Fusion
- 📌 **Citations** — every answer traces back to exact page + bounding box
- 🔌 **Provider-agnostic** — switch LLM/VLM via `.env` (Ollama, Gemini, OpenRouter, OpenAI)

---
## Cell 1: 🔧 Setup & Configuration

In [ ]:
# ============================================================
# SETUP — Install dependencies (run once)
# ============================================================
# Uncomment the line below if running outside Docker:
# !pip install -r ../requirements.txt

import sys
import os
import uuid
import json
import base64
import warnings
from pathlib import Path
from IPython.display import display, Markdown, HTML

warnings.filterwarnings("ignore")

# Add project root to path so we can import config & providers
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Load configuration ────────────────────────────────────────
from config.settings import settings
from providers.factory import get_llm, get_vlm, get_embeddings

# ── Show active configuration ─────────────────────────────────
print(settings.summary())

# ── Initialize provider objects ───────────────────────────────
print("\n⏳ Initializing providers...")
llm = get_llm()
vlm = get_vlm()
embeddings = get_embeddings()
print("✅ All providers initialized successfully!")
print(f"   LLM: {type(llm).__name__}")
print(f"   VLM: {type(vlm).__name__}")
print(f"   Embeddings: {type(embeddings).__name__}")

---
## Cell 2: 📊 Architecture Diagrams

### Ingestion Pipeline

```
┌─────────────────┐
│  User Uploads    │
│  PDF + selects   │
│  content type    │
└───────┬─────────┘
        │
        ▼
┌─────────────────────────────────────────────────────┐
│              Docling DocumentConverter               │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │  Layout   │ │   OCR    │ │  Table Structure   │  │
│  │ Analysis  │ │ (if scan)│ │   Recognition      │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
│              ┌──────────────┐                        │
│              │ Reading Order │                        │
│              └──────────────┘                        │
└───────────────────┬─────────────────────────────────┘
                    │
                    ▼
         ┌─────────────────────┐
         │  DoclingDocument     │
         │  (with provenance)   │
         └──────┬──────────────┘
                │
       ┌────────┼────────┐
       │        │        │
       ▼        ▼        ▼
  ┌─────────┐ ┌──────┐ ┌─────────────┐
  │  Text   │ │Table │ │   Image/    │
  │ Chunks  │ │Chunks│ │   Figure    │
  │(Hybrid) │ │(+hdr)│ │  → VLM      │
  └────┬────┘ └──┬───┘ │  summary    │
       │         │     └──────┬──────┘
       └─────────┼───────────┘
                 │
                 ▼
    ┌──────────────────────────────┐
    │  Chunk + Citation Metadata   │
    │  (page_no, bbox, headings,   │
    │   element_type, chunk_id)    │
    └──────┬──────────┬────────────┘
           │          │
           ▼          ▼
    ┌──────────┐ ┌──────────┐
    │ ChromaDB │ │  BM25    │
    │ (Vector) │ │ (Sparse) │
    └──────────┘ └──────────┘
```

### Retrieval + Answer Pipeline

```
┌──────────────┐
│  User Query  │
└──────┬───────┘
       │
  ┌────┴────┐
  │         │
  ▼         ▼
┌──────┐ ┌────────┐
│ BM25 │ │ChromaDB│
│(kw)  │ │(semant)│
└──┬───┘ └───┬────┘
   │         │
   └────┬────┘
        │
        ▼
┌────────────────────┐
│ Reciprocal Rank    │
│ Fusion (RRF)       │
│ EnsembleRetriever  │
└────────┬───────────┘
         │
         ▼
┌────────────────────┐
│ Top-K Results      │
│ + Citation Metadata│
└────────┬───────────┘
         │
         ▼
┌────────────────────┐
│ LLM Generation     │
│ (with [Source N]   │
│  inline citations) │
└────────┬───────────┘
         │
         ▼
┌────────────────────────────┐
│ Answer + Citation Cards    │
│ (page, bbox, heading,     │
│  excerpt, chunk_id)       │
└────────────────────────────┘
```

---
## Cell 3: 📄 PDF Upload & Content Type Selection

In [ ]:
# ============================================================
# PDF INPUT & CONTENT TYPE SELECTION
# ============================================================

# ── PDF path loaded from .env (can also be overridden here) ────
pdf_path = settings.resolved_pdf_path
PDF_PATH = str(pdf_path)

# ── Content type selection ────────────────────────────────────
# Options:
#   "text_only"   → Fast, no OCR, no table/image extraction
#   "tables"      → Enable table structure recognition
#   "images"      → Enable figure/picture extraction
#   "scanned"     → Enable full-page OCR
#   "mixed"       → Enable everything (tables + images + OCR)
#   "auto_detect" → Enable all features, let Docling decide

CONTENT_TYPE = "auto_detect"  # Change based on your PDF

print(f"📄 PDF Path (from .env): {PDF_PATH}")
print(f"📋 Content Type: {CONTENT_TYPE}")

# Validate the PDF exists
if not pdf_path.exists():
    print(f"\n⚠️  PDF not found at: {pdf_path.resolve()}")
    print("   Please update PDF_PATH in your .env file or place your PDF in the 'pdfs/' folder.")
else:
    print(f"✅ PDF found: {pdf_path.resolve()} ({pdf_path.stat().st_size / 1024:.1f} KB)")


---
## Cell 4: 🔍 Docling Extraction

In [ ]:
# ============================================================
# DOCLING EXTRACTION — Layout, OCR, Tables, Figures
# ============================================================

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.base_models import InputFormat

# ── Configure pipeline based on content type ─────────────────
pipeline_options = PdfPipelineOptions()

# Table structure recognition
if CONTENT_TYPE in ("tables", "mixed", "auto_detect"):
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
    print("✅ Table structure recognition: ENABLED (accurate mode)")
else:
    pipeline_options.do_table_structure = False
    print("⬜ Table structure recognition: DISABLED")

# OCR for scanned documents
if CONTENT_TYPE in ("scanned", "mixed", "auto_detect"):
    pipeline_options.do_ocr = True
    print("✅ OCR: ENABLED")
else:
    pipeline_options.do_ocr = False
    print("⬜ OCR: DISABLED")

# Picture/figure extraction
if CONTENT_TYPE in ("images", "mixed", "auto_detect"):
    pipeline_options.generate_picture_images = True
    print("✅ Picture extraction: ENABLED")
else:
    pipeline_options.generate_picture_images = False
    print("⬜ Picture extraction: DISABLED")

# ── Create converter and process ─────────────────────────────
print("\n⏳ Converting PDF with Docling...")
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

result = converter.convert(str(pdf_path))
doc = result.document

# ── Summary of extraction ────────────────────────────────────
print("\n" + "=" * 60)
print("📊 EXTRACTION SUMMARY")
print("=" * 60)

# Count element types
text_count = len(doc.texts) if hasattr(doc, 'texts') else 0
table_count = len(doc.tables) if hasattr(doc, 'tables') else 0
picture_count = len(doc.pictures) if hasattr(doc, 'pictures') else 0
page_count = len(doc.pages) if hasattr(doc, 'pages') else 0

print(f"   📄 Pages:    {page_count}")
print(f"   📝 Text blocks: {text_count}")
print(f"   📊 Tables:   {table_count}")
print(f"   🖼️  Pictures: {picture_count}")
print("=" * 60)

# Preview: export first 500 chars as markdown
md_preview = doc.export_to_markdown()[:500]
print("\n📖 Markdown Preview (first 500 chars):")
print("-" * 40)
print(md_preview)
print("-" * 40)

---
## Cell 5: 🖼️ Image Handling — Multimodal Summaries

In [ ]:
# ============================================================
# IMAGE HANDLING — Generate text summaries from figures
# ============================================================
# Strategy:
#   1. Extract images from the DoclingDocument
#   2. Send each image to the VLM (get_vlm()) for a text summary
#   3. Store the TEXT SUMMARY as a chunk (not the raw image)
#   4. Preserve citation metadata (page_no, bbox, "image_summary")
#
# The raw images are NOT stored in the vector store. Only their
# text descriptions are embedded and retrieved. Citations point
# back to the image location in the original PDF.
# ============================================================

from langchain_core.messages import HumanMessage
from PIL import Image
import io

image_summaries = []  # Will hold {text, metadata} dicts

pictures = doc.pictures if hasattr(doc, 'pictures') else []

if not pictures:
    print("ℹ️  No pictures found in the document. Skipping image summarization.")
else:
    print(f"🖼️  Found {len(pictures)} picture(s). Generating text summaries...\n")

    for idx, picture in enumerate(pictures):
        print(f"  Processing image {idx + 1}/{len(pictures)}...", end=" ")

        try:
            # ── Extract provenance metadata ────────────────────
            page_numbers = []
            bboxes = []
            if hasattr(picture, 'prov') and picture.prov:
                for prov in picture.prov:
                    page_numbers.append(prov.page_no)
                    if hasattr(prov, 'bbox') and prov.bbox:
                        bboxes.append({
                            "l": prov.bbox.l,
                            "t": prov.bbox.t,
                            "r": prov.bbox.r,
                            "b": prov.bbox.b,
                        })

            # ── Get the image data ────────────────────────────
            # Docling can export picture images if generate_picture_images=True
            image_data = None
            if hasattr(picture, 'image') and picture.image:
                # picture.image is a PIL Image or we can get bytes
                if hasattr(picture.image, 'pil_image'):
                    pil_img = picture.image.pil_image
                elif isinstance(picture.image, Image.Image):
                    pil_img = picture.image
                else:
                    print("⚠️ Unknown image format, skipping.")
                    continue

                # Convert to base64 for the VLM
                buf = io.BytesIO()
                pil_img.save(buf, format="PNG")
                image_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
                image_data = f"data:image/png;base64,{image_b64}"
            else:
                print("⚠️ No image data available, creating summary from caption only.")
                # Fallback: use caption if available
                caption = ""
                if hasattr(picture, 'caption_text'):
                    caption = picture.caption_text(doc)
                elif hasattr(picture, 'text'):
                    caption = picture.text

                if caption:
                    image_summaries.append({
                        "text": f"[Figure on page {page_numbers}]: {caption}",
                        "page_numbers": page_numbers,
                        "bboxes": bboxes,
                        "headings": [],
                        "element_type": "image_summary",
                        "chunk_id": str(uuid.uuid4()),
                    })
                    print(f"✅ (caption only)")
                else:
                    print("⏭️ No image data or caption.")
                continue

            # ── Send to VLM for text summary ──────────────────
            message = HumanMessage(
                content=[
                    {"type": "text", "text": (
                        "Describe this image/figure/diagram in detail. "
                        "Include all text, numbers, labels, and relationships shown. "
                        "If it's a chart or graph, describe the data trends. "
                        "If it's a diagram, describe the structure and connections."
                    )},
                    {"type": "image_url", "image_url": {"url": image_data}},
                ]
            )

            response = vlm.invoke([message])
            summary_text = response.content

            image_summaries.append({
                "text": f"[Figure on page {page_numbers}]: {summary_text}",
                "page_numbers": page_numbers,
                "bboxes": bboxes,
                "headings": [],
                "element_type": "image_summary",
                "chunk_id": str(uuid.uuid4()),
            })
            print(f"✅ ({len(summary_text)} chars)")

        except Exception as e:
            print(f"❌ Error: {e}")
            continue

print(f"\n📊 Generated {len(image_summaries)} image summary chunk(s).")

---
## Cell 6: ✂️ Chunking with Citation Metadata

In [ ]:
# ============================================================
# CHUNKING — HybridChunker with full citation provenance
# ============================================================
# HybridChunker is structure-aware:
#   - Respects document hierarchy (headings, sections)
#   - Keeps tables intact (with repeated headers for large tables)
#   - Merges small sibling chunks for better context
#   - Token-aligned to the embedding model
#
# For each chunk, we extract rich citation metadata:
#   - page_numbers: which page(s) the chunk spans
#   - bboxes: exact coordinates for PDF highlighting
#   - headings: section hierarchy for context
#   - element_type: "text", "table", or "image_summary"
#   - chunk_id: unique identifier for docstore lookup
# ============================================================

from docling.chunking import HybridChunker

# ── Initialize chunker (token-aligned to embedding model) ────
try:
    from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
    tokenizer = HuggingFaceTokenizer.from_pretrained(settings.EMBEDDING_MODEL)
    print(f"✅ Tokenizer loaded: {settings.EMBEDDING_MODEL}")
except Exception as e:
    print(f"⚠️  Could not load HuggingFace tokenizer: {e}")
    print("   Falling back to default tokenizer.")
    tokenizer = None

chunker_kwargs = {
    "max_tokens": settings.CHUNK_MAX_TOKENS,
    "merge_peers": settings.CHUNK_MERGE_PEERS,
}
if tokenizer:
    chunker_kwargs["tokenizer"] = tokenizer

chunker = HybridChunker(**chunker_kwargs)

# ── Chunk the document ────────────────────────────────────────
print(f"\n⏳ Chunking document (max_tokens={settings.CHUNK_MAX_TOKENS})...")
raw_chunks = list(chunker.chunk(doc))
print(f"   Generated {len(raw_chunks)} raw chunks from text/tables.")

# ── Extract citation metadata from each chunk ─────────────────
citation_chunks = []

for chunk in raw_chunks:
    # Build citation metadata
    page_numbers = []
    bboxes = []
    element_types = set()

    # Extract provenance from doc_items
    if hasattr(chunk, 'meta') and hasattr(chunk.meta, 'doc_items'):
        for item in chunk.meta.doc_items:
            # Get element type
            if hasattr(item, 'label'):
                element_types.add(str(item.label))

            # Get page numbers and bounding boxes
            if hasattr(item, 'prov'):
                for prov in item.prov:
                    if hasattr(prov, 'page_no'):
                        page_numbers.append(prov.page_no)
                    if hasattr(prov, 'bbox') and prov.bbox:
                        bboxes.append({
                            "l": float(prov.bbox.l),
                            "t": float(prov.bbox.t),
                            "r": float(prov.bbox.r),
                            "b": float(prov.bbox.b),
                        })

    # Get headings (section hierarchy)
    headings = []
    if hasattr(chunk, 'meta') and hasattr(chunk.meta, 'headings'):
        headings = list(chunk.meta.headings) if chunk.meta.headings else []

    # Determine primary element type
    if "table" in str(element_types).lower():
        primary_type = "table"
    elif "list" in str(element_types).lower():
        primary_type = "list"
    else:
        primary_type = "text"

    # Deduplicate page numbers
    page_numbers = sorted(set(page_numbers))

    citation_chunks.append({
        "text": chunk.text,
        "page_numbers": page_numbers,
        "bboxes": bboxes,
        "headings": headings,
        "element_type": primary_type,
        "chunk_id": str(uuid.uuid4()),
        "doc_id": str(chunk.meta.origin.binary_hash) if hasattr(chunk.meta, 'origin') and hasattr(chunk.meta.origin, 'binary_hash') else "unknown",
    })

# ── Append image summary chunks ───────────────────────────────
for img_chunk in image_summaries:
    if "doc_id" not in img_chunk:
        img_chunk["doc_id"] = citation_chunks[0]["doc_id"] if citation_chunks else "unknown"
    citation_chunks.append(img_chunk)

# ── Summary ───────────────────────────────────────────────────
print(f"\n" + "=" * 60)
print("✂️  CHUNKING SUMMARY")
print("=" * 60)
print(f"   📝 Text chunks:    {sum(1 for c in citation_chunks if c['element_type'] == 'text')}")
print(f"   📊 Table chunks:   {sum(1 for c in citation_chunks if c['element_type'] == 'table')}")
print(f"   📋 List chunks:    {sum(1 for c in citation_chunks if c['element_type'] == 'list')}")
print(f"   🖼️  Image summaries: {sum(1 for c in citation_chunks if c['element_type'] == 'image_summary')}")
print(f"   ─────────────────────")
print(f"   📦 Total chunks:   {len(citation_chunks)}")
print("=" * 60)

# ── Preview a chunk with its citation ─────────────────────────
if citation_chunks:
    sample = citation_chunks[0]
    print(f"\n📌 Sample Chunk (first):")
    print(f"   Text:     {sample['text'][:150]}...")
    print(f"   Pages:    {sample['page_numbers']}")
    print(f"   Headings: {sample['headings']}")
    print(f"   Type:     {sample['element_type']}")
    print(f"   BBoxes:   {len(sample['bboxes'])} region(s)")
    print(f"   Chunk ID: {sample['chunk_id']}")

---
## Cell 7: 📦 Indexing — Dual Store (Vector + BM25)

In [ ]:
# ============================================================
# INDEXING — ChromaDB (Vector) + BM25 (Sparse)
# ============================================================
# We create LangChain Document objects with full citation metadata,
# then index them into both stores for hybrid retrieval.
# ============================================================

from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.schema import Document

# ── Convert citation chunks to LangChain Documents ────────────
print("⏳ Converting chunks to LangChain Documents...")

lc_docs = []
for c in citation_chunks:
    if not c["text"].strip():
        continue  # Skip empty chunks

    # ChromaDB metadata must be flat (str, int, float, bool)
    # So we JSON-serialize complex fields
    lc_docs.append(Document(
        page_content=c["text"],
        metadata={
            "page_numbers": json.dumps(c["page_numbers"]),
            "bboxes": json.dumps(c["bboxes"]),
            "headings": json.dumps(c["headings"]),
            "element_type": c["element_type"],
            "chunk_id": c["chunk_id"],
            "doc_id": c.get("doc_id", "unknown"),
        },
    ))

print(f"   Created {len(lc_docs)} LangChain Documents.")

# ── Vector Store (ChromaDB) ───────────────────────────────────
print("\n⏳ Building ChromaDB vector index...")
vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=embeddings,
    collection_name="rag_citations",
    persist_directory="../data/chroma",
)
vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": settings.TOP_K}
)
print(f"   ✅ ChromaDB: {len(lc_docs)} documents indexed.")

# ── Sparse Store (BM25) ───────────────────────────────────────
print("\n⏳ Building BM25 sparse index...")
bm25_retriever = BM25Retriever.from_documents(lc_docs)
bm25_retriever.k = settings.TOP_K
print(f"   ✅ BM25: {len(lc_docs)} documents indexed.")

# ── DocStore (raw metadata for citation lookup) ───────────────
# Save the full citation data as JSON for downstream use
docstore_path = Path("../data/docstore/citations.json")
docstore_path.parent.mkdir(parents=True, exist_ok=True)
with open(docstore_path, "w") as f:
    json.dump(citation_chunks, f, indent=2, default=str)
print(f"\n   💾 DocStore saved: {docstore_path} ({len(citation_chunks)} entries)")

print("\n" + "=" * 60)
print("📦 INDEXING COMPLETE")
print("=" * 60)
print(f"   🔢 Vector Store: ChromaDB ({len(lc_docs)} docs)")
print(f"   🔤 Sparse Store: BM25 ({len(lc_docs)} docs)")
print(f"   💾 DocStore:     {docstore_path}")
print("=" * 60)

---
## Cell 8: 🔀 Hybrid Retrieval with RRF

In [ ]:
# ============================================================
# HYBRID RETRIEVAL — BM25 + Vector with Reciprocal Rank Fusion
# ============================================================
# EnsembleRetriever combines multiple retrievers using RRF.
# Weights control the influence of each retriever:
#   - Higher BM25 weight → better for exact keyword matches
#   - Higher Vector weight → better for semantic similarity
# ============================================================

from langchain.retrievers import EnsembleRetriever

# ── Create the hybrid retriever ───────────────────────────────
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[settings.BM25_WEIGHT, settings.VECTOR_WEIGHT],
)

print(f"🔀 Hybrid Retriever configured:")
print(f"   BM25 weight:   {settings.BM25_WEIGHT}")
print(f"   Vector weight: {settings.VECTOR_WEIGHT}")
print(f"   Top-K:         {settings.TOP_K}")


# ── Helper: Parse citation metadata from retrieved docs ───────
def parse_citation_metadata(doc: Document) -> dict:
    """Parse the JSON-serialized metadata back into Python objects."""
    return {
        "page_numbers": json.loads(doc.metadata.get("page_numbers", "[]")),
        "bboxes": json.loads(doc.metadata.get("bboxes", "[]")),
        "headings": json.loads(doc.metadata.get("headings", "[]")),
        "element_type": doc.metadata.get("element_type", "text"),
        "chunk_id": doc.metadata.get("chunk_id", ""),
        "doc_id": doc.metadata.get("doc_id", ""),
        "text": doc.page_content,
    }


# ── Test retrieval ────────────────────────────────────────────
TEST_QUERY = "What are the main topics discussed in this document?"

print(f"\n🔍 Test Query: \"{TEST_QUERY}\"")
print("-" * 60)

test_results = ensemble_retriever.invoke(TEST_QUERY)

for i, doc in enumerate(test_results[:3]):  # Show top 3
    citation = parse_citation_metadata(doc)
    print(f"\n📌 Result {i + 1}:")
    print(f"   Text:    {doc.page_content[:150]}...")
    print(f"   Pages:   {citation['page_numbers']}")
    print(f"   Section: {' > '.join(citation['headings']) if citation['headings'] else '(root)'}")
    print(f"   Type:    {citation['element_type']}")
    print(f"   BBoxes:  {len(citation['bboxes'])} region(s)")

print(f"\n✅ Retrieved {len(test_results)} results with full citation metadata.")

---
## Cell 9: 🤖 Answer Generation with Citations

In [ ]:
# ============================================================
# ANSWER GENERATION — LLM with inline citations
# ============================================================
# The prompt forces the LLM to:
#   1. Use ONLY the provided context
#   2. Cite every claim with [Source N]
#   3. List all sources with page numbers after the answer
# ============================================================

from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

# ── Citation-forcing prompt ───────────────────────────────────
CITATION_PROMPT_TEMPLATE = """You are a precise document search assistant. Your job is to answer questions based ONLY on the provided context from a PDF document.

CRITICAL RULES:
1. Every factual claim in your answer MUST have a citation in the format [Source N] where N matches the source number below.
2. If the context doesn't contain enough information to answer, say "The document does not contain sufficient information to answer this question."
3. Do NOT make up information or use knowledge outside the provided context.
4. After your answer, provide a CITATIONS section listing each source you used.

CONTEXT:
{context}

QUESTION: {question}

Provide your answer with inline [Source N] citations, followed by a CITATIONS section:"""


def format_context_with_sources(docs: list[Document]) -> str:
    """Format retrieved documents as numbered sources with metadata."""
    formatted_parts = []
    for i, doc in enumerate(docs):
        citation = parse_citation_metadata(doc)
        pages = citation["page_numbers"]
        headings = citation["headings"]
        element_type = citation["element_type"]

        header = f"[Source {i + 1}] (Page {pages}, Section: {' > '.join(headings) if headings else 'N/A'}, Type: {element_type})"
        formatted_parts.append(f"{header}\n{doc.page_content}")

    return "\n\n" + "---" + "\n\n".join(formatted_parts)


# ── Build the RAG chain ───────────────────────────────────────
prompt = ChatPromptTemplate.from_template(CITATION_PROMPT_TEMPLATE)


def retrieve_and_format(query: str) -> dict:
    """Retrieve documents and format them as context."""
    docs = ensemble_retriever.invoke(query)
    context = format_context_with_sources(docs)
    return {"context": context, "question": query, "_docs": docs}


def run_rag_query(query: str) -> dict:
    """
    Run the full RAG pipeline: retrieve → format → generate → cite.
    Returns the answer, retrieved docs, and structured citations.
    """
    # 1. Retrieve
    retrieved_docs = ensemble_retriever.invoke(query)

    # 2. Format context
    context = format_context_with_sources(retrieved_docs)

    # 3. Generate answer
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": query})

    return {
        "query": query,
        "answer": response.content,
        "retrieved_docs": retrieved_docs,
        "num_sources": len(retrieved_docs),
    }


# ── Run a test query ──────────────────────────────────────────
QUERY = "What are the main topics discussed in this document?"

print(f"🔍 Query: \"{QUERY}\"")
print("⏳ Running RAG pipeline...\n")

result = run_rag_query(QUERY)

print("=" * 60)
print("🤖 ANSWER:")
print("=" * 60)
print(result["answer"])
print("=" * 60)
print(f"\n📊 Used {result['num_sources']} source(s) for this answer.")

---
## Cell 10: 📋 Citation Display & Verification

In [ ]:
# ============================================================
# CITATION DISPLAY — Structured proof for every answer
# ============================================================
# This cell demonstrates how the final product would display
# citations. In production, these would render in a PDF viewer
# side panel with highlighted bounding boxes.
# ============================================================


def display_citations(rag_result: dict) -> list[dict]:
    """
    Display structured citation cards and return JSON for downstream use.

    In the final product, the JSON output drives the PDF viewer side panel:
    - page_numbers → scroll to page
    - bboxes → draw highlight overlay
    - headings → show breadcrumb navigation
    - text_excerpt → show preview snippet
    """
    query = rag_result["query"]
    answer = rag_result["answer"]
    docs = rag_result["retrieved_docs"]

    # ── Display the answer ────────────────────────────────────
    print("╔" + "═" * 58 + "╗")
    print("║  🔍 SEARCH RESULT WITH PROOF                              ║")
    print("╠" + "═" * 58 + "╣")
    print(f"║  Query: {query[:50]}{'...' if len(query) > 50 else ''}")
    print("╠" + "═" * 58 + "╣")
    print("║  ANSWER:")
    # Word-wrap the answer
    for line in answer.split("\n"):
        while len(line) > 56:
            print(f"║  {line[:56]}")
            line = line[56:]
        print(f"║  {line}")
    print("╠" + "═" * 58 + "╣")
    print("║  📌 CITATION EVIDENCE:")
    print("╠" + "═" * 58 + "╣")

    # ── Build structured citations ────────────────────────────
    citations_json = []

    for i, doc in enumerate(docs):
        citation = parse_citation_metadata(doc)

        print(f"║")
        print(f"║  ┌─ Source {i + 1} ────────────────────────────────────")
        print(f"║  │ 📄 Page(s):  {citation['page_numbers']}")
        print(f"║  │ 📂 Section:  {' > '.join(citation['headings']) if citation['headings'] else '(document root)'}")
        print(f"║  │ 🏷️  Type:     {citation['element_type']}")
        print(f"║  │ 📐 Regions:  {len(citation['bboxes'])} bounding box(es)")
        if citation['bboxes']:
            bb = citation['bboxes'][0]
            print(f"║  │             First: L={bb['l']:.1f} T={bb['t']:.1f} R={bb['r']:.1f} B={bb['b']:.1f}")
        excerpt = doc.page_content[:100].replace("\n", " ")
        print(f"║  │ 📝 Excerpt:  {excerpt}...")
        print(f"║  │ 🔑 ID:       {citation['chunk_id'][:16]}...")
        print(f"║  └──────────────────────────────────────────────")

        citations_json.append({
            "source_num": i + 1,
            "page_numbers": citation["page_numbers"],
            "bboxes": citation["bboxes"],
            "headings": citation["headings"],
            "element_type": citation["element_type"],
            "text_excerpt": doc.page_content[:200],
            "chunk_id": citation["chunk_id"],
        })

    print("║")
    print("╚" + "═" * 58 + "╝")

    return citations_json


# ── Display citations for the last query ──────────────────────
citations = display_citations(result)

# ── Output as JSON (for the PDF viewer side panel) ────────────
print("\n\n📋 Citation JSON (for PDF viewer integration):")
print("-" * 60)
print(json.dumps(citations, indent=2))

---
## Cell 11: ⚙️ Provider Info & Connectivity Test

In [ ]:
# ============================================================
# PROVIDER INFO — Verify all configured providers work
# ============================================================

print("⚙️  Provider Connectivity Test")
print("=" * 60)

# ── Test LLM ──────────────────────────────────────────────────
print(f"\n1️⃣  LLM ({settings.LLM_PROVIDER} / {settings.LLM_MODEL}):")
try:
    test_response = llm.invoke("Say 'hello' in one word.")
    print(f"   ✅ Response: {test_response.content[:50]}")
except Exception as e:
    print(f"   ❌ Error: {e}")

# ── Test VLM ──────────────────────────────────────────────────
print(f"\n2️⃣  VLM ({settings.VLM_PROVIDER} / {settings.VLM_MODEL}):")
try:
    test_response = vlm.invoke("Say 'vision ready' in two words.")
    print(f"   ✅ Response: {test_response.content[:50]}")
except Exception as e:
    print(f"   ❌ Error: {e}")

# ── Test Embeddings ───────────────────────────────────────────
print(f"\n3️⃣  Embeddings ({settings.EMBEDDING_PROVIDER} / {settings.EMBEDDING_MODEL}):")
try:
    test_embedding = embeddings.embed_query("test")
    print(f"   ✅ Dimension: {len(test_embedding)}")
except Exception as e:
    print(f"   ❌ Error: {e}")

print("\n" + "=" * 60)
print("\n💡 To switch providers, edit the .env file and restart the kernel.")
print("   Example: Change LLM_PROVIDER=ollama to LLM_PROVIDER=gemini")

---
## Cell 12: 🧪 Interactive Query Loop

In [ ]:
# ============================================================
# INTERACTIVE QUERY — Run multiple queries against the PDF
# ============================================================
# Change the QUERY variable and re-run this cell to search
# for different information in your PDF.
# ============================================================

QUERY = "Summarize the key findings of this document."  # ← Change this!

print(f"🔍 Query: \"{QUERY}\"")
print("⏳ Running RAG pipeline...\n")

result = run_rag_query(QUERY)
citations = display_citations(result)

# ── Save results ──────────────────────────────────────────────
query_result = {
    "query": QUERY,
    "answer": result["answer"],
    "citations": citations,
    "num_sources": result["num_sources"],
}

# Save to file for downstream use
results_path = Path("../data/docstore/last_query_result.json")
with open(results_path, "w") as f:
    json.dump(query_result, f, indent=2, default=str)

print(f"\n💾 Result saved to: {results_path}")
print("\n💡 Change the QUERY variable above and re-run this cell for a new search.")